# TinyMT32Gen — TinyMT-32 equidistribution

**TinyMT** (Saito-Matsumoto 2011) is a 127-bit-state MT variant
designed for parameter parallelism — each parameter set
$(\mathrm{mat}_1, \mathrm{mat}_2, \mathrm{tmat})$ defines an
independent generator, useful for spawning many uncorrelated streams.

This notebook uses the upstream default parameter set. Because the
state is small ($k = 127$, output $L = 32$) the matricial method
runs in milliseconds. See
[generators/TinyMT32Gen.md](../../generators/TinyMT32Gen.md).


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Saito & Matsumoto (2011)_


In [ ]:
# TinyMT-32 default parameter set.
gen = Generator.create("TinyMT32Gen", L=32,
    mat1=0x8F7011EE, mat2=0xFC78FF1F, tmat=0x3793FDFF)
print(gen.display())


## Wrap in a `Combination`


In [ ]:
# Wrap the generator in a single-component Combination. The
# Combination is the live object the search loop iterates and the
# equidistribution test consumes.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

TinyMT-32's $\chi_f$ is reducible (state $= 127$ bits but only $127 - 1 = 126$ are reachable from the typical seed); the **`notprimitive`** matricial-on-invariant-subspace method applies.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_NOTPRIMITIVE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "tinymt32-default"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('tinymt32-default')
# entry.components[0] carries the same params as constructed above
```
